In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import plotly.express as pxs
from tools.wiki_scrapper import get_bond_mod_names


In [2]:
bond_mods_list = get_bond_mod_names()
print(bond_mods_list)

['aerial_bond', 'astral_bond', 'contagious_bond', 'covert_bond', 'duplex_bond', 'manifold_bond', 'momentous_bond', 'mystic_bond', 'reinforced_bond', 'restorative_bond', 'seismic_bond', 'tandem_bond', 'tenacious_bond', 'vicious_bond']


In [8]:
df = pd.read_csv("../data/bond_90days_orders.csv", sep=';') 
display(df)

,Unnamed: 0,datetime,volume,min_price,max_price,open_price,closed_price,avg_price,wa_price,median,moving_avg,donch_top,donch_bot,id,mod_rank,mode,mod_name,order_type
0,283,2026-09-26T00:00:00.000+00:00,18,2.0,5.0,NaN,NaN,3.5,4.833,5.0,NaN,NaN,NaN,6ab86104e63ff80010bd3600,0,live,vicious_bond,buy
1,177,2026-09-26T00:00:00.000+00:00,73,13.0,17.0,16.0,15.0,15.0,14.932,15.0,14.8,20.0,9.0,6ab86102e63ff80010bd35bd,0,closed,momentous_bond,NaN
2,310,2026-09-26T00:00:00.000+00:00,1,10.0,10.0,NaN,NaN,10.0,10.000,10.0,NaN,NaN,NaN,6ab86103e63ff80010bd35dd,5,live,aerial_bond,buy
3,311,2026-09-26T00:00:00.000+00:00,13,3.0,5.0,NaN,NaN,4.0,4.846,4.0,NaN,NaN,NaN,6ab86103e63ff80010bd35d6,0,live,aerial_bond,buy
4,343,2026-09-26T00:00:00.000+00:00,1760,2.0,10.0,NaN,NaN,6.0,6.968,7.0,NaN,NaN,NaN,6ab86103e63ff80010bd35ea,0,live,manifold_bond,buy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6791,179,2026-06-30T00:00:00.000+00:00,59,5.0,12.0,NaN,NaN,8.5,7.153,10.0,NaN,NaN,NaN,6a4459b91e047a0012d26c13,0,live,tenacious_bond,buy
6792,179,2026-06-30T00:00:00.000+00:00,9,4.0,7.0,NaN,NaN,5.5,5.444,5.0,NaN,NaN,NaN,6a4459ba1e047a0012d26c40,0,live,contagious_bond,buy
6793,178,2026-06-30T00:00:00.000+00:00,55,3.0,5.0,NaN,NaN,4.0,4.709,5.0,NaN,NaN,NaN,6a4459ba1e047a0012d26c48,5,live,contagious_bond,buy
6794,1,2026-06-30T00:00:00.000+00:00,543,13.0,17.0,NaN,NaN,15.0,14.759,15.0,NaN,NaN,NaN,6a4459ba1e047a0012d26c3f,0,live,contagious_bond,sell


In [9]:

def clean_timeseries_data(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    print(f"size before the filter {df.shape[0]}")
    df = df[
        (df["mode"] == "live")
    ]
    print(f"size after the filter {df.shape[0]}")
    df = (
        df
        .groupby(["datetime", "mod_name", "mod_rank", "order_type"], as_index=False)
        .agg(
            min_price=("min_price", "mean"),
            avg_price=("avg_price", "mean"),
            max_price=("max_price", "mean"),
            volume=("volume", "sum"),
        )
        .sort_values(["mod_name", "datetime", "mod_rank", "order_type"])
    )
    print(f"size after the aggregation {df.shape[0]}")
    df["datetime"] = df["datetime"].dt.date
    return df

df = clean_timeseries_data(df)
display(df)

size before the filter 6796
size after the filter 4491
size after the aggregation 4491


,datetime,mod_name,mod_rank,order_type,min_price,avg_price,max_price,volume
0,2026-06-30,aerial_bond,0,buy,5.0,10.0,15.0,26
1,2026-06-30,aerial_bond,0,sell,17.0,18.5,20.0,1158
2,2026-06-30,aerial_bond,5,buy,6.0,6.0,6.0,1
3,2026-06-30,aerial_bond,5,sell,18.0,31.5,45.0,147
54,2026-07-01,aerial_bond,0,buy,5.0,10.0,15.0,31
...,...,...,...,...,...,...,...,...
4440,2026-09-25,vicious_bond,0,sell,9.0,14.0,19.0,310
4441,2026-09-25,vicious_bond,5,sell,14.0,20.0,26.0,212
4488,2026-09-26,vicious_bond,0,buy,2.0,3.5,5.0,18
4489,2026-09-26,vicious_bond,0,sell,10.0,15.0,20.0,359


In [10]:
bond_df = df[
    df["mod_name"].isin(bond_mods_list) &
    (df["mod_rank"] == 0)
].copy()

fig = px.line(
    bond_df,
    x="datetime",
    y="avg_price",
    color="mod_name",
    line_dash="order_type",
    markers=True,
    title="Bond Mods — Rank 0 — Buy vs Sell",
    labels={
        "datetime": "Date",
        "avg_price": "Average Price",
        "mod_name": "Mod",
        "order_type": "Order Type"
    }
)

fig.show()